# Ridge, Lasso and Elastic Net — learn by doing
Six lessons connect correlated housing measurements to coefficient preferences, conditional sparsity, fold-local selection and responsible comparison. Fictional embedded data only; no credentials, uploads or remote datasets. This trusted notebook is not the web evaluator and never awards mastery.

Predict before running each next cell; change only the guided authored examples locally. Reproducibility is not causal validity or proof of generalization. Genuine Colab execution remains UNVERIFIED.


In [ ]:
import sys, json, hashlib, warnings, importlib.metadata
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_absolute_error
profile = {"numpy":"2.5.4","pandas":"3.0.6","matplotlib":"3.11.2","scikit-learn":"1.9.1"}
assert sys.version_info[:3] == (3,13,16)
assert all(importlib.metadata.version(name)==version for name,version in profile.items())
SEED = 1729
np.random.seed(SEED)
print(json.dumps({"profile":profile,"seed":SEED},sort_keys=True))


## RG01 — why penalize?
Predict L1 and squared L2 for(1,−2). Is the intercept part of either? These terms express a preference, not validation error. A strong penalty may discard useful signal. Features with different units need training-only scaling.


In [ ]:
weights = np.array([1.,-2.])
l1 = np.abs(weights).sum()
l2_squared = weights @ weights
assert l1 == 3 and l2_squared == 5
ridge_objective = 4 + 2*l2_squared
lasso_objective = 4/(2*2) + .5*l1
mixture_penalty = .5*3 + .5*5/2
assert ridge_objective == 14 and lasso_objective == 2.5 and mixture_penalty == 2.75
print(json.dumps({"l1":float(l1),"l2_squared":float(l2_squared),"ridge":float(ridge_objective),"lasso":float(lasso_objective),"elastic_penalty":mixture_penalty},sort_keys=True))


## Exact fictional data and boundaries
Eight development-training houses and four external development-validation houses match the frozen experiments. Four additional final-test houses are withheld from all candidate decisions. Columns: area100m², room index, age decades; target: ten-thousand USD. Close area/room measurements motivate correlated-coefficient caution. No production dataset or human labels are collected.


In [ ]:
X = np.array([[1,1,2],[2,2,0],[3,2,1],[4,4,3],[5,4,1],[6,6,2],[7,6,0],[8,9,1]],dtype=float)
y = np.array([15,18,18,25,25,30,29,37],dtype=float)
X_validation = np.array([[1.5,2,1],[3.5,3,2],[6.5,7,1],[9,9,0]],dtype=float)
y_validation = np.array([16,22,31,36],dtype=float)
X_test = np.array([[2.5,2,1],[4.5,5,2],[7.5,8,1],[10,11,0]],dtype=float)
y_test = np.array([19,27,35,42],dtype=float)
train_ids = ["rg-h1","rg-h2","rg-h3","rg-h4","rg-h5","rg-h6","rg-h7","rg-h8"]
validation_ids = ["rg-v1","rg-v2","rg-v3","rg-v4"]
test_ids = ["rg-test1","rg-test2","rg-test3","rg-test4"]
bundle = {"train":[X.tolist(),y.tolist()],"validation":[X_validation.tolist(),y_validation.tolist()],"test":[X_test.tolist(),y_test.tolist()]}
dataset_sha256 = hashlib.sha256(json.dumps(bundle,sort_keys=True).encode()).hexdigest()
assert dataset_sha256 == "f0f73512cbc5769454cca9a11d373a4e8769794e92bf3185c7c88e7db5f18f03"
print(json.dumps({"rows":16,"features":3,"data_sha256":dataset_sha256},sort_keys=True))


## RG02/RG03 — predict shrinkage versus exact sparsity
Compare the same rows and units. Ridge uses SSE+α||w||²; Lasso uses SSE/(2n)+α||w||₁. Equal alpha is not equal strength. Exact zero means coefficient==0, not rounded0.00; near-zero nonzero is tagged separately. Do not infer causal irrelevance from sparsity or promise every individual weight/error decreases with alpha.


In [ ]:
def pipeline(kind, alpha, ratio=.5):
    if kind == "ridge":
        estimator = Ridge(alpha=alpha,solver="svd",tol=1e-10)
    elif kind == "lasso":
        estimator = Lasso(alpha=alpha,tol=1e-10,max_iter=100000,selection="cyclic",random_state=SEED)
    else:
        estimator = ElasticNet(alpha=alpha,l1_ratio=ratio,tol=1e-10,max_iter=100000,selection="cyclic",random_state=SEED)
    return make_pipeline(StandardScaler(),estimator)

def checked_fit(model, features, targets):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        model.fit(features,targets)
    assert not caught, [str(item.message) for item in caught]
    estimator=model.steps[-1][1]
    if hasattr(estimator,"dual_gap_"):
        assert estimator.dual_gap_ <= 1e-10*np.mean((targets-targets.mean())**2)+1e-12
    return model


In [ ]:
path=[]
for kind in ["ridge","lasso"]:
    for alpha in [.1,1,10]:
        fitted=checked_fit(pipeline(kind,alpha),X,y)
        coefficients=fitted.steps[-1][1].coef_
        tags=["exact-zero" if value==0 else "near-zero-nonzero" if abs(value)<=1e-8 else "nonzero" for value in coefficients]
        path.append({"model":kind,"alpha":alpha,"coefficients":coefficients.tolist(),"tags":tags,
            "training_mae":float(mean_absolute_error(y,fitted.predict(X))),
            "validation_mae":float(mean_absolute_error(y_validation,fitted.predict(X_validation)))})
assert np.allclose(path[0]["coefficients"],[2.0033726617649332,4.920277181455948,0.5533832481600581],atol=1e-9,rtol=1e-9)
print(json.dumps(path,sort_keys=True))


### Correlated-column comparison and debugging
Predict whether swapping columns AND their names must change predictions. Align feature semantics first. Repair a hypothetical warning by inspecting scaling, settings and convergence; never convert uncertain coefficients to zeros.


In [ ]:
original=checked_fit(pipeline("lasso",.1),X,y)
swapped=checked_fit(pipeline("lasso",.1),X[:,[1,0,2]],y)
assert np.allclose(original.predict(X_validation),swapped.predict(X_validation[:,[1,0,2]]),atol=1e-8,rtol=1e-8)
print(json.dumps({"aligned_swap_predictions_match":True,"causal_irrelevance_proved":False},sort_keys=True))


## RG04 — mixture is an objective, not an average of models
Predict the separate penalty components. Positive interior q values avoid treating unsupported pure-L2 ElasticNet fitting as a Ridge solver. Mixing does not guarantee superior error or stable feature identification.


In [ ]:
mixed=checked_fit(pipeline("elastic-net",1,.5),X,y)
coef=mixed.steps[-1][1].coef_
residual=y-mixed.predict(X)
terms={"data_loss":float(residual@residual/(2*len(y))),
       "l1_penalty":float(.5*np.abs(coef).sum()),
       "l2_penalty":float(.5*(coef@coef)/2)}
terms["objective"]=sum(terms.values())
assert all(np.isfinite(value) for value in terms.values())
print(json.dumps(terms,sort_keys=True))


## RG05 — complete pipeline inside each fold
Predict why fitting a scaler on all rows before CV leaks even without targets. Below each fold learns its own statistics; the mean of three MAEs is not pooled MAE when fold sizes differ. No final-test feedback chooses alpha, q or preprocessing. An unregularized baseline uses exactly the same CV protocol.


In [ ]:
folds=list(KFold(3,shuffle=True,random_state=SEED).split(X))
provenance=[]
for train_indices, held_indices in folds:
    fold_model=checked_fit(pipeline("elastic-net",.1,.5),X[train_indices],y[train_indices])
    assert np.allclose(fold_model.steps[0][1].mean_,X[train_indices].mean(axis=0))
    training=[train_ids[index] for index in train_indices]
    held=[train_ids[index] for index in held_indices]
    assert set(training).isdisjoint(held)
    provenance.append({"scaler_fit_ids":training,"estimator_fit_ids":training,"held_ids":held})
global_mean=np.vstack([X,X_validation]).mean(axis=0)
assert not np.allclose(global_mean,X.mean(axis=0))
print(json.dumps({"folds":provenance,"global_scaling_valid":False},sort_keys=True))


In [ ]:
candidates=[(kind,alpha,.5) for kind in ["ridge","lasso"] for alpha in [.1,1,10]]
candidates += [("elastic-net",alpha,ratio) for alpha in [.1,1] for ratio in [.25,.5,.75]]
scores=[]
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    for kind,alpha,ratio in candidates:
        model=pipeline(kind,alpha,ratio)
        mae=-cross_val_score(model,X,y,cv=folds,scoring="neg_mean_absolute_error",n_jobs=1)
        scores.append(float(mae.mean()))
assert not caught
baseline=make_pipeline(StandardScaler(),LinearRegression())
baseline_mae=float(-cross_val_score(baseline,X,y,cv=folds,scoring="neg_mean_absolute_error",n_jobs=1).mean())
chosen_index=int(np.argmin(scores))
chosen=candidates[chosen_index]
development_model=checked_fit(pipeline(*chosen),X,y)
validation_mae=float(mean_absolute_error(y_validation,development_model.predict(X_validation)))
print(json.dumps({"candidate_cv_mae":scores,"baseline_cv_mae":baseline_mae,"chosen":chosen,"external_validation_mae":validation_mae},sort_keys=True))


### Freeze selection, then final evaluation
Chosen settings are fixed using the eight-row CV comparison. External validation is reported but does not reselect. Refit the frozen workflow on the12 development rows, then evaluate the final test once. A toy test estimate is not production performance. Selecting a new candidate after seeing this score would invalidate the claimed final evaluation.


In [ ]:
frozen_choice=tuple(chosen)
final_model=checked_fit(pipeline(*frozen_choice),np.vstack([X,X_validation]),np.concatenate([y,y_validation]))
final_predictions=final_model.predict(X_test)
final_mae=float(mean_absolute_error(y_test,final_predictions))
assert tuple(chosen)==frozen_choice
print(json.dumps({"frozen_choice":frozen_choice,"final_test_ids":test_ids,"final_predictions":final_predictions.tolist(),"final_mae":final_mae},sort_keys=True))


## RG06 — restore units and compare responsibly
Before running, predict the raw slope for standardized coefficient6 and scale3. Adjust the intercept too. Compare error, exact sparsity and stability separately. A fixed seed is not sampling uncertainty; correlation and a regularized coefficient are not causal evidence. These models predict price, not calibrated binary churn/document quality.


In [ ]:
scaler,estimator=final_model.steps[0][1],final_model.steps[1][1]
raw_weights=estimator.coef_/scaler.scale_
raw_intercept=float(estimator.intercept_-raw_weights@scaler.mean_)
assert np.allclose(X_test@raw_weights+raw_intercept,final_predictions,atol=1e-9,rtol=1e-9)
assert 6/3==2 and 10+6*(7-4)/3 == 2+2*7 == 16
safe_order=("fold_training_scaler","fold_training_estimator")
assert safe_order != ("global_scaler","fold_training_estimator")
print(json.dumps({"rows":16,"folds":3,"candidates":12,"raw_weights":raw_weights.tolist(),"raw_intercept":raw_intercept,"final_mae":final_mae,"data_sha256":dataset_sha256,"mastery":"not awarded by notebook"},sort_keys=True))


## Reduced-scaffolding practice and report
1. Explain which normalization gives the Ridge14 versus Lasso2.5 examples.
2. Find a returned Lasso zero and distinguish it from near-zero nonzero; explain what it cannot establish.
3. Trace a single fold's scaler and estimator IDs; explain why the global diagnostic is invalid.
4. Compare the baseline and selected model without claiming a penalty always helps.
5. Report units, chosen settings, convergence, split provenance, uncertainty and population limits. Do not tune using the final test.

Expected structured checks are asserted above. No web mastery credit is awarded. Genuine screen-reader, zoom and Colab verification remain separate open obligations.
